# DS 3100 Case Study: Tennessee Education Data — Wrangling and Joins

**Goal:** Practice data-wrangling operations and learn why joining datasets is sometimes necessary to answer an analytical question.

This notebook intentionally stops **before EDA and visualization**. A separate exercise will use the resulting analytical data for exploratory analysis and visualization.

## 0. The datasets

We will work with two tables:

1. **`tenn2018(1).csv`** — Tennessee 2018 achievement data.
2. **`tn_district_info.csv`** — a small companion lookup table containing one row per district and a few district-level characteristics.

**Download the companion dataset:** [tn_district_info.csv](tn_district_info.csv)

In [ ]:
import pandas as pd

# Read the two datasets
# Update the paths if your files are stored elsewhere.
ten = pd.read_csv("tenn2018(1).csv")
district_info = pd.read_csv("tn_district_info.csv")

## 1. Get to know the achievement data

Before wrangling, identify the **unit of observation** and the variables that will drive the analysis.

In [ ]:
ten.shape
ten.columns.tolist()
ten.head()

### Think about the level of observation

The dataset contains statewide, district-level, and individual-school records. For this case study we want **district-level observations**.

- `district_number = 0`, `school_number = 0` → statewide aggregate
- `district_number > 0`, `school_number = 0` → district aggregate
- `district_number > 0`, `school_number > 0` → individual school

In [ ]:
# Keep district-level aggregates only
analysis = ten.loc[
    (ten["district_number"] > 0) &
    (ten["school_number"] == 0)
].copy()

analysis.head()

## 2. Define the analysis population

We want to compare achievement across subjects for the **All Students** subgroup.

Keep:

- `subgroup == "All Students"`
- ELA and Math observations

Then keep only the variables needed for the analysis.

In [ ]:
analysis = analysis.loc[
    (analysis["subgroup"] == "All Students") &
    (analysis["overall_subject"].isin(["ELA", "Math"]))
].copy()

analysis = analysis[[
    "district_number",
    "district_name",
    "overall_subject",
    "percent_below",
    "percent_on_mastered",
    "percent_below_previous",
    "percent_on_mastered_previous"
]]

analysis.head()

## 3. Create variables for change over time

Create:

- `change_below` = current `percent_below` − previous `percent_below`
- `change_on_mastered` = current `percent_on_mastered` − previous `percent_on_mastered`

Remember that the direction of a favorable change is different for the two measures.

In [ ]:
analysis["change_below"] = (
    analysis["percent_below"] - analysis["percent_below_previous"]
)

analysis["change_on_mastered"] = (
    analysis["percent_on_mastered"] - analysis["percent_on_mastered_previous"]
)

analysis.head()

## 4. Handle missing values deliberately

Before summarizing, inspect missingness rather than allowing it to remain invisible.

**Questions:**

- Which variables contain missing values?
- Why might a change variable be missing even when the current-year measure is available?

In [ ]:
analysis.isna().sum()

For the next summary, pandas' `mean()` and `median()` ignore missing values by default. This is different from filtering rows out of the dataset.

## 5. Group and summarize

For each subject, calculate:

- number of non-missing `percent_below` observations;
- mean `percent_below`;
- mean `percent_on_mastered`;
- mean `change_below`.

In [ ]:
subject_summary = (
    analysis.groupby("overall_subject")
    .agg(
        n_observations=("percent_below", "count"),
        mean_percent_below=("percent_below", "mean"),
        mean_percent_on_mastered=("percent_on_mastered", "mean"),
        mean_change_below=("change_below", "mean")
    )
    .reset_index()
)

subject_summary

## 6. Why do we need another dataset?

Suppose we now want to ask:

> **Do achievement outcomes differ across districts of different sizes?**

The achievement table does not contain the district-size classification we need. We therefore need information from another table.

The companion `tn_district_info.csv` is a deliberately small lookup table with **one row per district**.

In [ ]:
district_info.head()
district_info[["district_number", "district_name"]].duplicated().sum()

## 7. Identify the join key

The two tables share:

- `district_number`
- `district_name`

We will use the **pair of columns together** as a composite key. A match requires both values to agree.

We will use a **left join** because we want to preserve every observation in our achievement analysis and attach district information where a match exists.

In [ ]:
joined = analysis.merge(
    district_info,
    on=["district_number", "district_name"],
    how="left",
    validate="many_to_one"
)

joined.head()

## 8. Validate the join

A join can succeed syntactically and still be analytically wrong. Check what happened.

Ask:

1. Did the number of observations change?
2. Did every district receive a district-size value?
3. Did the join create unexpected missing values?

In [ ]:
print("Rows before join:", len(analysis))
print("Rows after join: ", len(joined))
print("Missing district_size:", joined["district_size"].isna().sum())

joined["district_size"].value_counts(dropna=False)

## 9. Post-join wrangling

Now that the district information is available, calculate the mean `percent_below` for each district-size category.

In [ ]:
size_summary = (
    joined.groupby("district_size", dropna=False)["percent_below"]
    .agg(n_observations="count", mean_percent_below="mean")
    .reset_index()
)

size_summary

## 10. Final wrangling challenge

Use the joined dataset to answer:

> **For ELA only, how does the average `percent_on_mastered` differ across district-size categories?**

Then create a second summary that compares the average `change_below` across the same categories.

### Before moving on

Be ready to explain:

- why we filtered to district-level observations;
- why `district_number` and `district_name` form a useful composite key;
- why we chose a left join; and
- how pandas handles missing values when calculating a mean.

In [ ]:
ela_size_summary = (
    joined.loc[joined["overall_subject"] == "ELA"]
    .groupby("district_size", dropna=False)
    .agg(
        mean_percent_on_mastered=("percent_on_mastered", "mean"),
        mean_change_below=("change_below", "mean")
    )
    .reset_index()
)

ela_size_summary

## 11. Wrap-up

At this point we have moved from raw tables to an analysis-ready dataset by:

**filtering → selecting → creating variables → handling missingness → grouping → summarizing → joining → validating the join → summarizing again**

The resulting `joined` data are ready for a separate **EDA and visualization** exercise.

In [ ]:
# Optional: save the joined analysis data for the next exercise
joined.to_csv("tn_joined_analysis.csv", index=False)